### CellPoseSAM


### Preparing the environment

In [ ]:
!pip install git+https://www.github.com/mouseland/cellpose.git

In [ ]:
!pip install tifffile

In [ ]:
import numpy as np
from pathlib import Path
from tqdm import trange
import matplotlib.pyplot as plt
import tifffile as tiff
from ipywidgets import interact, IntSlider
from skimage.transform import resize

In [ ]:
import cellpose
from cellpose import models, utils, core, io, plot, transforms
from cellpose.utils import outlines_list

In [ ]:
io.logger_setup() # run this to get printing of progress

#Check if colab notebook instance has GPU access
if core.use_gpu()==False:
  raise ImportError("No GPU access, change your runtime")

model = models.CellposeModel(gpu=True)

Upload and visualize images

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

# Directories of the images and labels
img_dir_F_GYR = Path("/content/drive/MyDrive/February_full/GYR_images")
img_dir_F_BYR = Path("/content/drive/MyDrive/February_full/BYR_images")

In [ ]:
# Load DAPI+Bcat+ZO-1 (BYR) images
image_files_BYR = sorted(img_dir_F_BYR.glob("*.tif"))
assert len(image_files_BYR) == 5, f"5 images were expected, but there are {len(image_files_BYR)}"

imgs_BYR = [tiff.imread(str(f)) for f in image_files_BYR]

In [ ]:
# Load Pha+Bcat+ZO-1 (GYR) images
image_files_GYR = sorted(img_dir_F_GYR.glob("*.tif"))
assert len(image_files_GYR) == 5, f"5 images were expected, but there are {len(image_files_GYR)}"

imgs_GYR = [tiff.imread(str(f)) for f in image_files_GYR]

### Check the images are correctly loaded

In [ ]:
image_files_BYR
# Observe metadata of the images to confirm they are correct
for i, f_path in enumerate(image_files_BYR):
    with tiff.TiffFile(str(f_path)) as f:
        print(f"Imagen {i+1}")
        print("  Series shape :", f.series[0].shape)
        print("  Series axes  :", f.series[0].axes)
        print("  Dtype        :", f.series[0].dtype)

In [ ]:
image_files_GYR
# Observe metadata of the images to confirm they are correct
for i, f_path in enumerate(image_files_GYR):
    with tiff.TiffFile(str(f_path)) as f:
        print(f"Imagen {i+1}")
        print("  Series shape :", f.series[0].shape)
        print("  Series axes  :", f.series[0].axes)
        print("  Dtype        :", f.series[0].dtype)

Each image has a different number of z-slides

### Running the tool

In [ ]:

print(cellpose.version)

# 3D segmentation

Function to segment in 3D, considering all XY, YZ, XZ planes

In [ ]:
def segment_3D(img, scale=1, z_step=1, diameter=20,
                          gpu=True, flow3D_smooth=1, batch_size=8,
                          pixel_size_xy=1.0, voxel_depth=1.0,
                          flow_threshold=0.4, cellprob_threshold=0.0, niter=None):

    anisotropy = voxel_depth / pixel_size_xy
    print(f"Anisotropy calculated: {anisotropy:.3f}")

    model = models.CellposeModel(gpu=gpu)
    volumen = img[::z_step, :, :, :]

    if scale != 1.0:
        H, W = volumen.shape[1:]
        H_new, W_new = int(H * scale), int(W * scale)
        volumen = resize(volumen, (volumen.shape[0], H_new, W_new),
                         preserve_range=True).astype(np.float32)

        print(f"Shape volumen processed: {volumen.shape} ({volumen.nbytes / 1e6:.1f} MB)")

    mask, flows, styles = model.eval(
        volumen,
        z_axis=0,
        channel_axis=1,
        batch_size=batch_size,
        diameter=diameter,
        do_3D=True,
        flow3D_smooth=flow3D_smooth,
        anisotropy=anisotropy,
        flow_threshold=flow_threshold,
        cellprob_threshold=cellprob_threshold,
        niter=niter)

    print(f"Segmentation completed | {mask.max()} labels detected")
    return mask

Segmentation for all images

In [ ]:
out_dir = Path("/content/drive/MyDrive/February_full/masks_pre")

for i, img in enumerate(imgs_BYR):

    print(f"\nSegmenting image {i+1}/{len(imgs_BYR)}")

    mask = segment_3D(
    imgs_BYR[i], diameter=70,
    scale=1, z_step=1,
    pixel_size_xy=0.6, voxel_depth=0.3,
    flow_threshold=0.2,
    cellprob_threshold=-3.5,
    niter=1200,
    flow3D_smooth=[12,1,1])

    # Save the resulting labels
    save_path = out_dir / f"mask_BYR_1.31_{i+1}.tif"
    tiff.imwrite(str(save_path), mask.astype(np.uint16))

    print(f"Saved: {save_path}")

In [ ]:
out_dir = Path("/content/drive/MyDrive/February_full/masks_pre")

for i, img in enumerate(imgs_GYR):

    print(f"\nSegmenting image {i+1}/{len(imgs_GYR)}")

    mask = segment_3D(
    imgs_GYR[i], diameter=70,
    scale=1, z_step=1,
    pixel_size_xy=0.6, voxel_depth=0.3,
    flow_threshold=0.2,
    cellprob_threshold=-1,
    niter=800,
    flow3D_smooth=[7,3,3])

    # Save the resulting labels
    save_path = out_dir / f"mask_GYR_2.10.1_{i+1}.tif"
    tiff.imwrite(str(save_path), mask.astype(np.uint16))

    print(f"Saved: {save_path}")

### Intersect between labels

In [ ]:
def intersect_labels(mask_dapi, mask_pha):
    """
    - mask_dapi: good XY extension, defines cell frontiers (Z, H, W)
    - mask_pha : good Z extension, defines cell volume (Z, H, W)
    """
    valid_region = mask_pha > 0  # True where there is a cell in Pha

    mask_final = mask_dapi.copy()
    mask_final[~valid_region] = 0 

    print(f"Original labels (DAPI): {mask_dapi.max()} | "
          f"Cells after intersection: {mask_final.max()}")
    return mask_final

In [ ]:
for i in range(len(imgs_BYR)):
  mask_dapi = tiff.imread(
      f"/content/drive/MyDrive/February_full/masks_pre/mask_BYR_1.31_{i+1}.tif") #load BYR
  mask_pha = tiff.imread(
      f"/content/drive/MyDrive/February_full/masks_pre/mask_GYR_2.10.1_{i+1}.tif") #load GYR
  mask_final = intersect_labels(mask_dapi, mask_pha) #intersection

  out_dir = Path("/content/drive/MyDrive/February_full/masks_intersect")
  save_path = out_dir / f"mask_intersect_{i+1}.tif"
  tiff.imwrite(str(save_path), mask_final.astype(np.uint16)) #save resulting labels
